# Session 6 — Creating and Deploying Containerized ML Applications using Docker, Flask/FastAPI, and Kubernetes on Google Cloud

**Goal:** package a trained model as a container image, run it locally with Docker,
then deploy that same image to a Kubernetes cluster on Google Cloud (GKE) — the
standard path from "works on my laptop" to a scalable, self-healing production
service.

## The three layers

1. **The app** — a small Flask/FastAPI service wrapping a trained model behind a
   `/predict` endpoint (Session 7 goes deeper on API design specifically).
2. **The container** — a `Dockerfile` that packages the app, its dependencies, and the
   model artifact into one portable image. Runs identically on your laptop and in the
   cloud.
3. **The orchestrator** — Kubernetes schedules containers across a cluster, restarts
   them if they crash, and scales replicas up/down under load.

## Prerequisites

* **Docker Desktop** (or another local Docker daemon) to build and run the image —
  this notebook writes the files and shows the exact commands, but does not execute
  `docker build`/`docker run` in-sandbox since no Docker daemon is available here.
* **A GKE cluster** (needs a GCP project with billing) for the final deployment step.
```bash
pip install flask
```

## Step 1 — Train and save a small model

The same iris classifier from Session 1, saved to disk so the container can load it
without needing to retrain.

In [ ]:
import os, joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

os.makedirs("session6_app", exist_ok=True)

X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=100, random_state=0).fit(X, y)
joblib.dump(model, "session6_app/model.joblib")
print("Saved session6_app/model.joblib")

## Step 2 — Write the Flask serving app

A minimal `/predict` endpoint: accept a JSON list of feature vectors, return
predicted classes. `/health` lets Kubernetes (or a load balancer) check the container
is alive before routing traffic to it.

In [ ]:
app_py = '''\
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
model = joblib.load("model.joblib")

@app.route("/health")
def health():
    return {"status": "ok"}

@app.route("/predict", methods=["POST"])
def predict():
    payload = request.get_json()
    X = np.array(payload["instances"])
    preds = model.predict(X).tolist()
    return jsonify({"predictions": preds})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)
'''
with open("session6_app/app.py", "w") as f:
    f.write(app_py)
print(app_py)

## Step 3 — Write the Dockerfile

Each instruction is a cached layer: dependencies are installed before the app code is
copied in, so re-building after an app-only change reuses the (slow) dependency
install layer instead of repeating it.

In [ ]:
dockerfile = '''\
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py model.joblib ./

EXPOSE 8080
CMD ["python", "app.py"]
'''
with open("session6_app/Dockerfile", "w") as f:
    f.write(dockerfile)

with open("session6_app/requirements.txt", "w") as f:
    f.write("flask\nscikit-learn\njoblib\nnumpy\n")

print(dockerfile)

## Step 4 — Build and run the image locally

Run these from a terminal with Docker Desktop running (not executed here — no Docker
daemon in this sandbox):

In [ ]:
build_and_run = '''\
cd session6_app
docker build -t iris-classifier:v1 .
docker run -p 8080:8080 iris-classifier:v1

# In another terminal:
curl -X POST http://localhost:8080/predict \\
  -H "Content-Type: application/json" \\
  -d '{"instances": [[5.1, 3.5, 1.4, 0.2]]}'
# -> {"predictions": [0]}
'''
print(build_and_run)

## Step 5 — Push the image to a registry

Kubernetes pulls images from a registry, not your local Docker cache — Google
Artifact Registry is the GCP-native option.

In [ ]:
push_commands = '''\
gcloud auth configure-docker us-central1-docker.pkg.dev
docker tag iris-classifier:v1 us-central1-docker.pkg.dev/YOUR_PROJECT/ml-images/iris-classifier:v1
docker push us-central1-docker.pkg.dev/YOUR_PROJECT/ml-images/iris-classifier:v1
'''
print(push_commands)

## Step 6 — Kubernetes manifests: Deployment + Service

A **Deployment** tells Kubernetes how many replicas of the container to keep running
and how to roll out updates. A **Service** gives those replicas a stable network
address and load-balances traffic across them.

In [ ]:
deployment_yaml = '''\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-classifier
spec:
  replicas: 3
  selector:
    matchLabels:
      app: iris-classifier
  template:
    metadata:
      labels:
        app: iris-classifier
    spec:
      containers:
        - name: iris-classifier
          image: us-central1-docker.pkg.dev/YOUR_PROJECT/ml-images/iris-classifier:v1
          ports:
            - containerPort: 8080
          readinessProbe:
            httpGet:
              path: /health
              port: 8080
            initialDelaySeconds: 5
          resources:
            requests:
              cpu: "250m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: iris-classifier-service
spec:
  type: LoadBalancer
  selector:
    app: iris-classifier
  ports:
    - port: 80
      targetPort: 8080
'''
with open("session6_app/deployment.yaml", "w") as f:
    f.write(deployment_yaml)
print(deployment_yaml)

## Step 7 — Deploy to GKE

In [ ]:
deploy_commands = '''\
gcloud container clusters create ml-cluster --num-nodes=3 --region=us-central1
gcloud container clusters get-credentials ml-cluster --region=us-central1

kubectl apply -f session6_app/deployment.yaml
kubectl get pods                  # watch 3 replicas come up
kubectl get service iris-classifier-service   # get the external IP once provisioned
'''
print(deploy_commands)
print("Replicas=3 means: if one pod crashes or a node dies, Kubernetes reschedules it")
print("automatically -- the readinessProbe stops traffic reaching a pod before it's ready.")

## What to try next

* Add a `HorizontalPodAutoscaler` targeting CPU utilization, so replica count scales
  with traffic instead of being fixed at 3.
* Session 7 dives deeper into the API layer itself (request validation, error
  handling, versioning) — swap the bare Flask app here for that FastAPI design.
* Session 10 automates Steps 4-5 (build, tag, push) inside a GitHub Actions workflow
  triggered on every merge to `main`.